# Phase 5: Spark Performance Tuning

Answers the challenge from notebook 04: is `Window(partitionBy="pickup_borough")`
a skew problem? Then covers the core levers professionals reach for:
shuffle partitions, caching, and Adaptive Query Execution (AQE).

**Docs:** [SQL Performance Tuning Guide](https://spark.apache.org/docs/latest/sql-performance-tuning.html) | [Adaptive Query Execution](https://spark.apache.org/docs/latest/sql-performance-tuning.html#adaptive-query-execution) | [`DataFrame.cache`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.cache.html)

## Concept: data skew vs. "shuffle is just slow"

A shuffle redistributes rows across partitions using a hash of the
partitioning key. If the *key's distribution* is uneven — e.g. `Manhattan`
has ~200x the rows of `Staten Island` — then hash partitioning sends wildly
different row counts to different partitions. Spark parallelizes across
tasks, but a stage only finishes when its *slowest* task finishes. One
partition with 4M rows next to nine partitions with a few hundred thousand
each means most executors sit idle waiting on the one straggler. This is
**skew**, and it's a distinct problem from "shuffles are expensive" — you
can have a small, fast shuffle that's still skewed, and a huge evenly-split
shuffle that parallelizes perfectly fine.

## Concept: `spark.sql.shuffle.partitions`

Every `groupBy`/`join`/`Window.partitionBy` triggers a shuffle into this many
partitions (default **200** — tuned for cluster-scale data, way too many for
a single-machine 9M-row dataset, which creates task-scheduling overhead with
tiny partitions). We've already been setting it to `8` in every notebook's
setup cell to match `local[*]`'s core count on this machine. The real-world
rule of thumb: aim for partitions that are large enough to amortize task
overhead (~100-200MB per partition is a common target) but numerous enough
to use all available cores.

## Concept: caching / persisting

`.cache()` (alias for `.persist(StorageLevel.MEMORY_AND_DISK)`) materializes
a DataFrame after its first action, so repeated reuse skips recomputing the
full lineage. Without it, **every action re-runs the entire DAG from the
source files** — lazy evaluation means nothing is free just because you
"already computed" a DataFrame in an earlier cell. Cache when you'll trigger
multiple actions against the same DataFrame; skip it for single-use ones (the
cache itself costs memory + a pass to populate).

## Concept: Adaptive Query Execution (AQE)

AQE (`spark.sql.adaptive.enabled`, **on by default since Spark 3.2**)
re-optimizes the physical plan *during* execution using actual runtime
statistics instead of only static estimates. Three sub-features matter most:
- **Coalescing shuffle partitions**: merges many small post-shuffle
  partitions into fewer, right-sized ones — reduces the "200 tiny tasks"
  problem without hand-tuning `shuffle.partitions` per query.
- **Skew join handling** (`spark.sql.adaptive.skewJoin.enabled`): detects an
  oversized partition in a sort-merge join and splits it into smaller
  sub-partitions automatically.
- **Switching join strategies at runtime**: e.g. converting a sort-merge join
  to a broadcast join if a filtered intermediate result turns out smaller
  than expected — the static plan can't know this in advance, AQE can.

This is why you've seen `AdaptiveSparkPlan isFinalPlan=false` at the top of
every `.explain()` output so far — the "final" plan isn't decided until
execution, precisely because AQE may still rewrite it.


In [ ]:
# Setup: same read/cast/clean/join pipeline from notebooks 01-04, now
# imported from the dataforge_ai package instead of duplicated inline.
import os
import time
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from dataforge_ai import read_trips, clean_trips, join_zones

spark = (
    SparkSession.builder
    .appName("DataForge-Phase5-Performance")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Effective driver memory:", spark.conf.get("spark.driver.memory"))

RAW = "../data/raw"
assert os.path.exists(RAW), f"Can't find {RAW}. Current dir is: {os.getcwd()}"

paths = [
    f"{RAW}/yellow_tripdata_2023-01.parquet",
    f"{RAW}/yellow_tripdata_2023-02.parquet",
    f"{RAW}/yellow_tripdata_2023-03.parquet",
]
trips_clean = clean_trips(read_trips(spark, paths))

zones = (
    spark.read.option("header", True).option("inferSchema", True)
    .csv(f"{RAW}/taxi_zone_lookup.csv")
)

trips_with_zones = join_zones(trips_clean, zones).withColumn(
    "pickup_hour", F.hour("tpep_pickup_datetime")
)

print("trips_with_zones:", trips_with_zones.count(), "rows")

trips_with_zones: 9301798 rows


## Worked example: measuring the borough skew from notebook 04

`partitionBy("pickup_borough")` has only 8 distinct keys (7 boroughs + the
CSV's `N/A` placeholder). With `spark.sql.shuffle.partitions=8`, that's a
close-to-worst-case skew setup: each borough's rows land almost entirely in
their own partition (hash collisions aside), and we already saw the row
counts range from 260 (Staten Island) to 4M+ (Manhattan). Let's confirm this
by inspecting actual per-partition row counts after the shuffle.


In [ ]:
# Repartition by the same key the Window function uses, then inspect how
# many rows landed in each physical partition via spark_partition_id().
by_borough = trips_with_zones.repartition(8, "pickup_borough")

skew_check = (
    by_borough
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .agg(
        F.count("*").alias("row_count"),
        F.collect_set("pickup_borough").alias("boroughs_in_partition"),
    )
    .orderBy(F.desc("row_count"))
)
skew_check.show(truncate=False)

# Compare against an evenly-distributed repartition (no key, just round-robin).
even_check = (
    trips_with_zones.repartition(8)
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .agg(F.count("*").alias("row_count"))
    .orderBy(F.desc("row_count"))
)
even_check.show()


## Worked example: caching cost/benefit

`trips_with_zones` is reused across many cells in notebooks 04-05 — a
prime caching candidate. Let's measure the actual wall-clock difference
between recomputing from source vs. reading from cache.


In [2]:
def time_action(label: str, fn):
    start = time.time()
    result = fn()
    elapsed = time.time() - start
    print(f"{label}: {elapsed:.2f}s -> {result}")
    return elapsed

# Without cache: every .count()/.agg() below re-reads + re-cleans + re-joins
# from the raw parquet files, because trips_with_zones' lineage isn't
# materialized anywhere.
print("--- uncached ---")
time_action("count #1", lambda: trips_with_zones.count())
time_action("count #2", lambda: trips_with_zones.count())
time_action("avg fare", lambda: trips_with_zones.agg(F.avg("fare_amount")).first()[0])

# With cache: first action triggers computation AND materialization; every
# subsequent action reads from the cached in-memory/disk representation.
trips_cached = trips_with_zones.cache()
print("\n--- cached ---")
time_action("count #1 (populates cache)", lambda: trips_cached.count())
time_action("count #2 (reads from cache)", lambda: trips_cached.count())
time_action("avg fare (reads from cache)", lambda: trips_cached.agg(F.avg("fare_amount")).first()[0])

trips_cached.unpersist()  # free the memory once we're done demonstrating


--- uncached ---
count #1: 4.75s -> 9301798
count #2: 4.57s -> 9301798
avg fare: 4.83s -> 18.84067249364111

--- cached ---
count #1 (populates cache): 13.88s -> 9301798
count #2 (reads from cache): 0.11s -> 9301798
avg fare (reads from cache): 0.17s -> 18.840672493643044


DataFrame[VendorID: bigint, tpep_pickup_datetime: timestamp, tpep_dropoff_datetime: timestamp, passenger_count: double, trip_distance: double, RatecodeID: double, store_and_fwd_flag: string, PULocationID: bigint, DOLocationID: bigint, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, airport_fee: double, pickup_zone: string, pickup_borough: string, dropoff_zone: string, dropoff_borough: string, pickup_hour: int]

## Hands-on task: does AQE actually rescue the borough-skew window query?

Build `compare_aqe(trips)` that runs the notebook-04 `demand_rank` window
query (busiest hour per borough) **twice** — once with AQE off, once with
AQE on — and returns the wall-clock time for each, plus whether the final
(post-execution) physical plan differs. AQE's plan only "settles" once the
query actually runs (`isFinalPlan=true`), so we trigger an action before
calling `.explain()` to see the real post-adaptive plan, not the initial
static one.


In [3]:
def build_demand_query(trips: DataFrame) -> DataFrame:
    """The notebook-04 busiest-hour-per-borough window query, isolated so we
    can re-run it under different AQE settings.
    """
    hourly = (
        trips
        .groupBy("pickup_borough", "pickup_hour")
        .agg(F.count("*").alias("trip_count"))
    )
    rank_window = Window.partitionBy("pickup_borough").orderBy(F.desc("trip_count"))
    return hourly.withColumn("demand_rank", F.dense_rank().over(rank_window))


def compare_aqe(trips: DataFrame) -> dict:
    """Run the same window query with AQE off vs. on, timing each and
    capturing whether the *executed* physical plan differs.

    `.explain()` alone only shows the static/initial plan. To see AQE's real
    rewritten plan we must trigger an action first (`.collect()`), THEN call
    `.explain()` -- Spark's AdaptiveSparkPlan caches the final executed plan
    on the DataFrame's query execution once adaptive re-optimization has
    actually run.
    """
    results = {}
    for aqe_enabled in (False, True):
        spark.conf.set("spark.sql.adaptive.enabled", str(aqe_enabled).lower())
        query = build_demand_query(trips)

        start = time.time()
        query.collect()
        elapsed = time.time() - start

        key = "aqe_on" if aqe_enabled else "aqe_off"
        results[key] = {"seconds": elapsed}
        print(f"\n--- AQE {'ON' if aqe_enabled else 'OFF'}: {elapsed:.2f}s ---")
        query.explain()

    spark.conf.set("spark.sql.adaptive.enabled", "true")  # restore default
    return results


timings = compare_aqe(trips_with_zones)
print("\nSummary:", timings)



--- AQE OFF: 5.18s ---
== Physical Plan ==
Window [dense_rank(trip_count#4093L) windowspecdefinition(pickup_borough#1559, trip_count#4093L DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS demand_rank#4099], [pickup_borough#1559], [trip_count#4093L DESC NULLS LAST]
+- *(9) Sort [pickup_borough#1559 ASC NULLS FIRST, trip_count#4093L DESC NULLS LAST], false, 0
   +- Exchange hashpartitioning(pickup_borough#1559, 8), ENSURE_REQUIREMENTS, [plan_id=1921]
      +- *(8) HashAggregate(keys=[pickup_borough#1559, pickup_hour#1661], functions=[count(1)])
         +- Exchange hashpartitioning(pickup_borough#1559, pickup_hour#1661, 8), ENSURE_REQUIREMENTS, [plan_id=1917]
            +- *(7) HashAggregate(keys=[pickup_borough#1559, pickup_hour#1661], functions=[partial_count(1)])
               +- *(7) Project [pickup_borough#1559, hour(tpep_pickup_datetime#97, Some(Etc/UTC)) AS pickup_hour#1661]
                  +- *(7) BroadcastHashJoin [DOLocationID#237L]

## Further reading

- [SQL Performance Tuning Guide](https://spark.apache.org/docs/latest/sql-performance-tuning.html) — shuffle partitions, join strategies, AQE
- [Adaptive Query Execution](https://spark.apache.org/docs/latest/sql-performance-tuning.html#adaptive-query-execution) — skew join handling, partition coalescing, runtime join-strategy switching
- [`DataFrame.persist` / `StorageLevel`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.persist.html) — memory vs. disk tradeoffs beyond the default `.cache()`
- [Spark Web UI Guide](https://spark.apache.org/docs/latest/web-ui.html) — the SQL/Stages tabs at `localhost:4040` show real task-level timing/skew, which is the ground-truth tool for anything measured here with `time.time()`

**Note:** examples built against PySpark 3.5.3 (this container's version);
the official docs above may reference newer APIs (latest is 4.2.0 as of this
writing) — check version-specific behavior if something doesn't match.
